# IMDB review- Sentiment analysis- LSTM

In [1]:
!pip install kaggle

# Importing Dependencies

In [2]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Loading Data

In [3]:
data=pd.read_csv(r"D:\Materials\Academic\Master's\Research\PdM\Code\Deep Learning\IMDB Dataset.csv")

In [4]:
data.shape

(50000, 2)

In [5]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [6]:
data.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [7]:
data['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [8]:
# Label encoding
data.replace({"sentiment": {"positive": 1, "negative":0}}, inplace=True)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_4556\1992319307.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({"sentiment": {"positive": 1, "negative":0}}, inplace=True)


In [9]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [10]:
# Split the data
train_data, test_data=train_test_split(data, test_size=0.2, random_state=42)

In [11]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


# Data preprocessing

In [12]:
# Initialize a tokenizer that will keep the top 5000 most frequent words
tokenizer = Tokenizer(num_words=5000)

# Build the word index by fitting the tokenizer on the training reviews
tokenizer.fit_on_texts(train_data["review"])

# Convert training reviews into sequences of integers and pad them to length 200
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)

# Convert test reviews into sequences of integers and pad them to length 200
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)


In [13]:
print(X_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [14]:
print(X_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [15]:
# Decide Y data
Y_train=train_data["sentiment"]
Y_test=test_data["sentiment"]

In [16]:
print(Y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


# Building the LSTM model

In [20]:
# Initialize a sequential model to stack layers in order
model = Sequential()

# Add an embedding layer to convert word indices into dense 128-dimensional vectors
model.add(Embedding(input_dim=5000, output_dim=128))

# Add an LSTM layer with 128 units and dropout for regularization
#dropout=0.2, recurrent_dropout=0.2 → Regularization to prevent overfitting by randomly dropping connections during training.
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# Add a dense output layer with sigmoid activation for binary classification
model.add(Dense(1, activation='sigmoid'))

# Build the model with the expected input shape (batch size unspecified, sequence length = 200)
model.build(input_shape=(None, 200))

In [21]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (None, 200, 128)            │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_2 (LSTM)                        │ (None, 128)                 │         131,584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 1)                   │             129 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

Param or parameter number is the multiplication of input*output size of a layer (640000=5000*128)
Output None mean no batch size

In [24]:
# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Training the model

In [26]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 113s 218ms/step - accuracy: 0.7820 - loss: 0.4641 - val_accuracy: 0.8360 - val_loss: 0.3924
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 119s 238ms/step - accuracy: 0.8473 - loss: 0.3597 - val_accuracy: 0.8094 - val_loss: 0.4169
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 111s 223ms/step - accuracy: 0.8633 - loss: 0.3313 - val_accuracy: 0.8664 - val_loss: 0.3323
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 114s 227ms/step - accuracy: 0.8877 - loss: 0.2821 - val_accuracy: 0.8643 - val_loss: 0.3237
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 117s 235ms/step - accuracy: 0.9040 - loss: 0.2443 - val_accuracy: 0.8586 - val_loss: 0.3436


# Model Evaluation

In [27]:
loss, accuracy= model.evaluate(X_test, Y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.8653 - loss: 0.3300
Test Loss: 0.3300431966781616
Test Accuracy: 0.8652999997138977


# Build a predictive system

In [28]:
def predict_sentiment(review):
  # tokenize and pad the review
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence, maxlen=200)
  prediction = model.predict(padded_sequence)
  sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
  return sentiment

In [29]:
# example usage
new_review = "This movie was fantastic. I loved it."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 339ms/step
The sentiment of the review is: positive


In [30]:
# example usage
new_review = "This movie was not that good"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
The sentiment of the review is: negative


In [31]:
# example usage
new_review = "This movie was ok but not that good."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
The sentiment of the review is: negative
